# How to Fine-Tune a Decision Model with Unsloth (SFT + LoRA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_for_decision_models_with_unsloth.ipynb)

This tutorial shows you how to fine-tune a decision model, a model that makes decisions instead of generating text, on your own labels.

We fine-tune our [`d1-3B`](https://huggingface.co/LiquidAI/d1-3B) decision model with Supervised Fine-tuning (SFT) and LoRA using Unsloth, with intent classification for customer support routing on Banking77 as the example.

Follow along if it's your first time using Unsloth, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **LoRA + SFT** - Fine-tune the d1-3B decision model on your own decisions using LoRA (from PEFT)
- **Evaluation** - Compare the decision model's accuracy before and after fine-tuning

## 📋 Prerequisites:
- **GPU Runtime**: Select a GPU in `Runtime` → `Change runtime type`. A free T4 GPU is enough (about 30 minutes of training)
- **Hugging Face Account**: Only needed to push your model to the Hub

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==5.17.0 "torchao>0.16.0"
!pip install --no-deps trl==1.15.0

### Loading d1-3B

d1-3B ships its own code for the System One API (`system_one` and `system_one_batch`), so we load it with `trust_remote_code = True`.

We also download the model repository locally. It contains `prompt.py`, the module d1-3B uses to turn a state and a question into a prompt. We will reuse it to build our training data, so that the model sees the exact same prompt during training and inference.

Unsloth needs to be imported before Transformers, so we import it first.

In [2]:
from unsloth import FastVisionModel # Import Unsloth before Transformers!
import torch
import sys
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

model_id = "LiquidAI/d1-3B"

# bfloat16 on GPUs that support it natively (e.g. A100), float16 otherwise (e.g. T4)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation = False) else torch.float16

# Download the checkpoint and d1-3B's own code, and make its prompt builder importable
local_dir = snapshot_download(model_id)
sys.path.insert(0, local_dir)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- hybrid.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- prompt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- api.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- lfm2_vl.py
- runner.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions 

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

### What is a decision model?

A decision model is a model that makes decisions instead of generating text: it answers yes/no, multiple-choice, and scoring questions with a probability for each answer, in a single forward pass. Typical uses are classification, routing and triage, content moderation, and LLM-as-a-judge scoring.

d1-3B is a 3B parameter decision model built on [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B). It takes a state and a set of questions, and returns typed answers in one forward pass, without generating any output tokens. Each question uses one of the following three primitives:

- `noul`: Yes or no, as P(yes)
- `choice`: One of named options, with a probability for each
- `score`: A level on an ordered scale of 2 to 10 levels

Before we fine-tune, let's see how d1-3B makes a decision. We ask a `choice` question about a customer message:

In [3]:
state = "I was charged twice this month, please refund one of them."

questions = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
}

response = model.system_one(state = state, questions = questions)

print(response)

{'answers': {'team': {'type': 'choice', 'choice': 'billing', 'confidence': 0.982860184342666, 'probabilities': {'billing': 0.982860184342666, 'technical': 0.01091859038543887, 'fraud': 0.0062212252718952575}}}, 'usage': {'input_tokens': 61, 'output_tokens': 0}}


Under the hood, d1-3B renders the state and the question into a single prompt. Each option gets a short code (`A`, `B`, `C`, ...), and the prompt ends right after the assistant header:

In [4]:
from prompt import aliases, as_question, render

rendered_prompt = render(
    tokenizer = tokenizer,
    state = state,
    q = as_question(questions["team"]),
    bos = tokenizer.bos_token,
)

print(rendered_prompt)

<|startoftext|><|im_start|>user
I was charged twice this month, please refund one of them.


QUESTION:
Which team should handle this?

Options:
A Charges, refunds, invoices
B App or site faults
C Suspected unauthorised use

Reply with the option code only.<|im_end|>
<|im_start|>assistant



The model then runs one forward pass and reads the next-token probabilities at the end of the prompt, the **answer slot**. It only looks at the tokens of the option codes and normalizes them into the probabilities you saw above.

This means that fine-tuning a decision model is regular SFT where **the completion is a single token: the code of the correct option**, followed by the end-of-turn token `<|im_end|>`.

Let's free up the GPU memory before we load the model for training:

In [5]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

### Unsloth

For training, we load the same weights with Unsloth's `FastVisionModel`. d1-3B's checkpoint uses the standard LFM2-VL module names, so we don't need its custom code here.

**Why not `FastDecisionModel`?** Unsloth also has `FastDecisionModel` and `DecisionTrainer` for training decision models. Given d1-3B, they would add a new, randomly initialized decision head on top of the model and save the result in Unsloth's own decision-model format. You would lose d1-3B's trained readout and its System One API. Instead, we fine-tune d1-3B's own readout: the model keeps answering at the answer slot, and the fine-tuned model works with `system_one` exactly like d1-3B.

In [6]:
from unsloth import FastVisionModel
from transformers import AutoModelForImageTextToText

model, processor = FastVisionModel.from_pretrained(
    model_name = model_id,
    # d1-3B's config only registers its own AutoModel class, so we name the standard class with the language model head
    auto_model = AutoModelForImageTextToText,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False, # 4 bit quantization to reduce memory
)

==((====))==  Unsloth 2026.10.6: Fast Lfm2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Lfm2DecoderLayer).


We now add LoRA adapters so we only need to update a small amount of parameters!

Our task is text-only, so we only adapt the language model: its attention layers (`q_proj`, `k_proj`, `v_proj`, `out_proj`), its short convolution layers (`in_proj`, `out_proj`), and its MLP layers (`w1`, `w2`, `w3`). Unsloth's `finetune_*_layers` options select layers by name and skip LFM2's convolution layers, so we pass a regex as `target_modules` instead.

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    # Only the language model's layers, not the vision encoder's
    target_modules = r".*language_model.*\.(q_proj|k_proj|v_proj|out_proj|in_proj|w1|w2|w3)",
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We will use [Banking77](https://huggingface.co/datasets/mteb/banking77), a dataset of ~13k online banking customer messages, each labeled with one of 77 fine-grained intents. We train on 3,000 messages and evaluate on 100 to keep things fast. Feel free to increase these numbers with `select(range(...))`.

In [8]:
from datasets import load_dataset
dataset = load_dataset("mteb/banking77")
train_dataset = dataset["train"].shuffle(seed = 42).select(range(3000))
test_dataset = dataset["test"].shuffle(seed = 42).select(range(100))
dataset

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Let's see how the first row looks like!

In [9]:
train_dataset[0]

{'text': 'How long is a top up supposed to be pending, because an hour is too long! Can you fix this to go through?',
 'label': 47,
 'label_text': 'pending_top_up'}

### Data processing

Banking77 contains reworded versions of the same request, such as "can you tell me why my transfer failed?" and "Can you please tell me why my transfer failed?". If one lands in training and the other in evaluation, the evaluation rewards memorization and overstates what fine-tuning gained.

So we split by **case**: messages that ask the same thing stay on the same side. If your own data has a case, ticket, or conversation ID, group by that. Banking77 has none, so we approximate one: two messages belong to the same case when they share at least 80% of their words.

In [10]:
import random
from collections import Counter

def words(text):
    return set(re.findall(r"[a-z0-9']+", text.lower()))

def same_case(a, b, threshold = 0.8):
    """Whether two messages' word sets overlap enough to count as rewordings of one request."""
    union = a | b
    return bool(union) and len(a & b) / len(union) >= threshold

def case_ids(texts):
    """Give messages that reword each other the same case ID, transitively."""
    word_sets = [words(text) for text in texts]
    parent = list(range(len(texts)))

    def find(i):
        while parent[i] != i:
            i = parent[i]
        return i

    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            if same_case(word_sets[i], word_sets[j]):
                parent[find(j)] = find(i)
    return [find(i) for i in range(len(texts))]

# 1. Drop training messages that reword a test message
test_words = [words(text) for text in test_dataset["text"]]
leaked = {i for i, text in enumerate(train_dataset["text"]) if any(same_case(words(text), w) for w in test_words)}
train_dataset = train_dataset.select([i for i in range(len(train_dataset)) if i not in leaked])

# 2. Hold out 10% of the training messages to track the loss, taking whole cases only
cases = case_ids(train_dataset["text"])
case_sizes = Counter(cases)
shuffled_cases = sorted(case_sizes)
random.Random(42).shuffle(shuffled_cases)
eval_cases, n_eval = set(), 0
for case in shuffled_cases:
    if n_eval >= 0.1 * len(train_dataset):
        break
    eval_cases.add(case)
    n_eval += case_sizes[case]

sft_train = train_dataset.select([i for i, case in enumerate(cases) if case not in eval_cases])
sft_eval = train_dataset.select([i for i, case in enumerate(cases) if case in eval_cases])

print(f"Dropped {len(leaked)} training messages that reword a test message")
print(f"{len(cases) - len(case_sizes)} training messages reword another training message")
print(f"No case on both sides: {len(sft_train)} train and {len(sft_eval)} eval messages")

Dropped 6 training messages that reword a test message
77 training messages reword another training message
No case on both sides: 2694 train and 300 eval messages


We turn the classification task into a single `choice` question. Each intent becomes one option, described by its label name.

**Important**: The option codes are assigned by position: the first option gets the first code, the second option the second, and so on. If we always trained with the options in the same order, the model could learn which code means which intent without reading the option list. Adding or reordering an intent later would then shift the codes and break the fine-tuned model. We avoid that below by giving every training example its own random subset of the intents, in a random order.

In [11]:
labels = sorted(set(dataset["train"]["label_text"]))

intent_question = {
    "type": "choice",
    "instructions": "Which intent does this banking customer message express?",
    "criteria": {label: label.replace("_", " ") for label in labels},
}

# The code d1-3B assigns to each option in this order, e.g. "card_arrival" -> "12"
codes = {label: code for label, (code, _) in zip(labels, aliases(tokenizer, labels))}

print(f"Number of options: {len(labels)}")
print(f"First option codes: {dict(list(codes.items())[:5])}")

Number of options: 77
First option codes: {'Refund_not_showing_up': '00', 'activate_my_card': '01', 'age_limit': '02', 'apple_pay_or_google_pay': '03', 'atm_support': '04'}


With up to 26 options, d1-3B uses the letters `A` to `Z` as codes. With more options, as here, it switches to the two-digit codes `00` to `76`, each of which is a single token. Going from 26 to 27 options therefore changes every code at once, which is another reason to train on shuffled option lists. Our subsets keep at least 27 options, so the codes stay two-digit, as they are when we ask with all 77 intents.

We now render every message into d1-3B's prompt with its own option list: the correct intent plus a random number of the other intents (27 to 77 options in total), in a random order. We append the code the correct intent gets in that list, followed by `<|im_end|>`, and save it to `text`.

Because the list changes from example to example, a code says nothing about which intent it stands for: the model has to read the option list to find the right one, whichever intents it contains.

In [12]:
def formatting_prompts_func(example, idx, split):
    # Every example gets its own option list: the correct intent plus 26 to 76 others, in a random order
    rng = random.Random(f"{split}-{idx}")
    others = [label for label in labels if label != example["label_text"]]
    options = rng.sample(others, rng.randint(26, len(others))) + [example["label_text"]]
    rng.shuffle(options)
    question = as_question({**intent_question, "criteria": {label: intent_question["criteria"][label] for label in options}})
    option_codes = {label: code for label, (code, _) in zip(options, aliases(tokenizer, options))}
    rendered_prompt = render(
        tokenizer = tokenizer,
        state = example["text"],
        q = question,
        bos = tokenizer.bos_token,
    )
    return { "text" : rendered_prompt + option_codes[example["label_text"]] + tokenizer.eos_token }

# Each example gets one option list, which is enough for one epoch. To train for more epochs, map the dataset
# once per epoch with a different `split` value and concatenate the copies, so every pass sees new lists.
train_dataset = sft_train.map(formatting_prompts_func, with_indices = True, fn_kwargs = {"split": "train"}, remove_columns = sft_train.column_names)
eval_dataset = sft_eval.map(formatting_prompts_func, with_indices = True, fn_kwargs = {"split": "eval"}, remove_columns = sft_eval.column_names)

Map:   0%|          | 0/2694 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [13]:
print(train_dataset[0]["text"])

<|startoftext|><|im_start|>user
Whats the process of tranfering money into an account?


QUESTION:
Which intent does this banking customer message express?

Options:
00 reverted card payment?
01 card acceptance
02 cash withdrawal charge
03 card swallowed
04 declined cash withdrawal
05 get physical card
06 wrong exchange rate for cash withdrawal
07 request refund
08 wrong amount of cash received
09 automatic top up
10 transaction charged twice
11 apple pay or google pay
12 top up by bank transfer charge
13 edit personal details
14 card payment not recognised
15 card linking
16 verify my identity
17 supported cards and currencies
18 extra charge on statement
19 compromised card
20 age limit
21 terminate account
22 top up by card charge
23 card delivery estimate
24 visa or mastercard
25 direct debit payment not recognised
26 atm support
27 order physical card
28 cash withdrawal not recognised
29 exchange charge
30 balance not updated after bank transfer
31 why verify identity
32 lost or s

Let's look at two training messages with the same intent. Each prompt has its own option list, so `card arrival` gets a different code in each, and the model has to read the list to answer.

In [14]:
# The first two training messages whose intent is card_arrival
for i in [i for i, label in enumerate(sft_train["label_text"]) if label == "card_arrival"][:2]:
    text = train_dataset[i]["text"]
    options = re.findall(r"^\d\d .+$", text, re.M)
    completion = text.rsplit("<|im_start|>assistant\n", 1)[1]
    print(f"Message: {sft_train[i]['text']}")
    print(f"Number of options: {len(options)}")
    print(f"First options: {options[:3]}")
    print(f"Correct option: {next(option for option in options if option.endswith(' card arrival'))}")
    print(f"Completion: {completion!r}\n")

Message: Where is the card I ordered 2 weeks ago?
Number of options: 76
First options: ['00 get disposable virtual card', '01 edit personal details', '02 getting virtual card']
Correct option: 08 card arrival
Completion: '08<|im_end|>'

Message: Can I track when my card will be delivered?
Number of options: 69
First options: ['00 activate my card', '01 getting spare card', '02 getting virtual card']
Correct option: 55 card arrival
Completion: '55<|im_end|>'



<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We train for one full epoch. Our data is text-only, so we pass the text tokenizer instead of the processor.

In [15]:
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 0.05, # A float is a fraction of the total steps
        num_train_epochs = 1, # Set this for 1 full training run.
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2694 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. For a decision model, the assistant output is the option code.

In [16]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/2694 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the first row again.

In [17]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

<|startoftext|><|im_start|>user
Whats the process of tranfering money into an account?


QUESTION:
Which intent does this banking customer message express?

Options:
00 reverted card payment?
01 card acceptance
02 cash withdrawal charge
03 card swallowed
04 declined cash withdrawal
05 get physical card
06 wrong exchange rate for cash withdrawal
07 request refund
08 wrong amount of cash received
09 automatic top up
10 transaction charged twice
11 apple pay or google pay
12 top up by bank transfer charge
13 edit personal details
14 card payment not recognised
15 card linking
16 verify my identity
17 supported cards and currencies
18 extra charge on statement
19 compromised card
20 age limit
21 terminate account
22 top up by card charge
23 card delivery estimate
24 visa or mastercard
25 direct debit payment not recognised
26 atm support
27 order physical card
28 cash withdrawal not recognised
29 exchange charge
30 balance not updated after bank transfer
31 why verify identity
32 lost or s

Now let's print the masked out example - you should see only the option code and `<|im_end|>`:

In [18]:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ").strip())

36<|im_end|>


In [19]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
5.934 GB of memory reserved.


In [20]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,694 | Num Epochs = 1 | Total steps = 169
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,461,312 of 3,147,945,200 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.178654,0.208320
100,0.205857,0.175606
150,0.151148,0.161431
169,0.171742,0.160396


Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Lfm2VlForConditionalGeneration's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


In [21]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

267.9952 seconds used for training.
4.47 minutes used for training.
Peak reserved memory = 7.125 GB.
Peak reserved memory for training = 1.191 GB.
Peak reserved memory % of max memory = 18.041 %.
Peak reserved memory for training % of max memory = 3.016 %.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit, scroll down!

In [22]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

### Saving to float16

To use the System One API with your fine-tuned model, merge the LoRA adapters into the model and save it in `float16`. The merged checkpoint contains only the weights and the tokenizer, so we also copy d1-3B's own code and processor configuration into the same folder.

In [23]:
import shutil

output_dir = "d1-3B-banking77-merged"

# Merge to 16bit
model.save_pretrained_merged(output_dir, tokenizer, save_method = "merged_16bit",)

# d1-3B's code for the System One API, its processor configuration, and its license
for filename in ["api.py", "hybrid.py", "lfm2_vl.py", "modeling_d1.py", "prompt.py", "runner.py", "processor_config.json", "LICENSE"]:
    shutil.copy(os.path.join(local_dir, filename), output_dir)

# To push the merged model and d1-3B's code to the Hugging Face Hub, uncomment:
# from huggingface_hub import create_repo, upload_folder
# create_repo("your-model-path", exist_ok = True)
# upload_folder(folder_path = output_dir, repo_id = "your-model-path")

Unsloth: Restored added_tokens_decoder metadata in d1-3B-banking77-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `d1-3B-banking77-merged`: 100%|██████████| 1/1 [00:14<00:00, 14.14s/it]


Successfully copied all 1 files from cache to `d1-3B-banking77-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:20<00:00, 20.35s/it]


Unsloth: Merge process complete. Saved to `/content/d1-3B-banking77-merged`


<a name="Inference"></a>
### Evaluation
Let's see how fine-tuning changed d1-3B's decisions. We first free up the GPU memory and measure how well the original d1-3B performs on the 100 test messages. `system_one_batch` packs many messages into each forward pass.

In [24]:
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from tqdm.auto import tqdm

def evaluate(model, dataset, batch_size = 16):
    """Accuracy of the model's intent decisions on `dataset`, and its predictions."""
    predictions = []
    # Evaluate in batches, so the progress bar shows how far we are
    for start in tqdm(range(0, len(dataset), batch_size), desc = "Evaluating"):
        texts = dataset[start:start + batch_size]["text"]
        responses = model.system_one_batch([(text, {"intent": intent_question}) for text in texts])
        predictions += [response["answers"]["intent"]["choice"] for response in responses]
    accuracy = sum(p == y for p, y in zip(predictions, dataset["label_text"])) / len(dataset)
    return accuracy, predictions

base_model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

baseline_accuracy, baseline_predictions = evaluate(base_model, test_dataset)
print(f"Baseline accuracy: {baseline_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

Baseline accuracy: 83.0%


Now let's free up the GPU memory again and load the fine-tuned model the same way: with `AutoModel` and `trust_remote_code = True`.

In [25]:
del base_model
gc.collect()
torch.cuda.empty_cache()

finetuned = AutoModel.from_pretrained(
    output_dir,
    trust_remote_code = True,
    dtype = dtype,
).to("cuda")

finetuned_accuracy, finetuned_predictions = evaluate(finetuned, test_dataset)
print(f"Baseline accuracy:   {baseline_accuracy:.1%}")
print(f"Fine-tuned accuracy: {finetuned_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

Baseline accuracy:   83.0%
Fine-tuned accuracy: 90.0%
